# MSD Special State Hybrid Prototype

This notebook prototypes the special-state MSD construction in a way that uses the Gemini logical path where it is currently supported, and drops to a physical squin prefix where it is not.

The key compiler limitation is that Bloqade-Lanes currently applies `logical_initialization` one Steane block at a time, so it cannot natively prepare a 5-block entangled fake input before encoding. Because of that, this notebook does the following:

1. Build the **logical MSD suffix** in the Gemini logical dialect and compile it so we can reuse its detector/observable post-processing.
2. Build a **physical 35-qubit prefix** that prepares the special state by acting on the five injection qubits before encoding.
3. Measure the 35 physical qubits and feed those raw shots through the logical suffix post-processing.

This lets us test the hybrid idea honestly, without pretending the current compiler can already splice the entangled prefix through `logical_initialization`.

In [1]:
import sys
from pathlib import Path

import numpy as np

REPO_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
for candidate in REPO_ROOT_CANDIDATES:
    candidate = candidate.resolve()
    if (candidate / 'demo' / 'msd_utils').exists():
        REPO_ROOT = candidate
        break
else:
    raise FileNotFoundError('Could not locate the bloqade-lanes repo root.')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from bloqade import qubit, squin
from bloqade.gemini import logical as gemini_logical
from bloqade.gemini.logical.stdlib import default_post_processing
from bloqade.lanes import GeminiLogicalSimulator
from bloqade.squin import kernel
from bloqade.tsim import Circuit

from demo.msd_utils.circuits import build_task

print('repo root:', REPO_ROOT)

repo root: /Users/jasonhan/Desktop/qmain/kirin-workspace/bloqade-lanes


## Logical Suffix

These kernels are the part we can still express directly in the Gemini logical dialect: logical MSD followed by basis tomography on output logical qubit `0`.

We compile them with `default_post_processing(reg)` and then reuse the resulting post-processing on raw shots from the physical special-state kernel.

In [2]:
@kernel
def logical_msd_forward(reg):
    squin.broadcast.sqrt_x([reg[0], reg[1], reg[4]])
    squin.broadcast.cz([reg[0], reg[2]], [reg[1], reg[3]])
    squin.broadcast.sqrt_y([reg[0], reg[3]])
    squin.broadcast.cz([reg[0], reg[3]], [reg[2], reg[4]])
    squin.sqrt_x_adj(reg[0])
    squin.broadcast.cz([reg[0], reg[1]], [reg[4], reg[3]])
    squin.broadcast.sqrt_x_adj(reg)


@gemini_logical.kernel(aggressive_unroll=True)
def logical_suffix_x():
    reg = qubit.qalloc(5)
    logical_msd_forward(reg)
    squin.h(reg[0])
    return default_post_processing(reg)


@gemini_logical.kernel(aggressive_unroll=True)
def logical_suffix_y():
    reg = qubit.qalloc(5)
    logical_msd_forward(reg)
    squin.sqrt_z_adj(reg[0])
    squin.h(reg[0])
    return default_post_processing(reg)


@gemini_logical.kernel(aggressive_unroll=True)
def logical_suffix_z():
    reg = qubit.qalloc(5)
    logical_msd_forward(reg)
    return default_post_processing(reg)


sim = GeminiLogicalSimulator()
logical_tasks = {
    'X': build_task(sim, logical_suffix_x, m2dets=None, m2obs=None, append_measurements=False),
    'Y': build_task(sim, logical_suffix_y, m2dets=None, m2obs=None, append_measurements=False),
    'Z': build_task(sim, logical_suffix_z, m2dets=None, m2obs=None, append_measurements=False),
}

for basis, task in logical_tasks.items():
    print(basis, 'logical suffix ready; detector-error-model observables =', task.detector_error_model.num_observables)

X logical suffix ready; detector-error-model observables = 5
Y logical suffix ready; detector-error-model observables = 5
Z logical suffix ready; detector-error-model observables = 5


In [10]:
logical_tasks["Z"].noiseless_tsim_circuit.diagram(height=500)

## Physical Prefix

This is the physical special-state prototype using output logical qubit `0`, as requested.

The structure is:

1. Prepare the basis-dependent tomography state on the distinguished input physical qubit.
2. Apply the physical 5-qubit MSD inverse on the five injection qubits.
3. Encode each 7-qubit Steane block while preserving the already-prepared input state on qubit `6` of each block.
4. Apply the logical MSD circuit transversally.
5. Apply the final logical tomography transversally on block `0`.
6. Measure all 35 physical qubits.

The measurement shots are then interpreted using the logical suffix post-processing from above.

In [ ]:
@kernel
def steane7_encode_existing(qubits):
    squin.broadcast.sqrt_y_adj(qubits[:6])
    evens = qubits[::2]
    odds = qubits[1::2]
    squin.broadcast.cz(odds, evens[1:])
    squin.sqrt_y(qubits[6])
    squin.broadcast.cz(evens[:-1], [qubits[3], qubits[5], qubits[6]])
    squin.broadcast.sqrt_y(qubits[2:])
    squin.broadcast.cz(evens[:-1], odds)
    squin.broadcast.sqrt_y([qubits[1], qubits[2], qubits[4]])
    squin.x(qubits[3])
    squin.broadcast.z([qubits[1], qubits[5]])


@kernel
def physical_msd_inverse(inputs):
    squin.broadcast.sqrt_x(inputs)
    squin.broadcast.cz([inputs[0], inputs[1]], [inputs[4], inputs[3]])
    squin.sqrt_x(inputs[0])
    squin.broadcast.cz([inputs[0], inputs[3]], [inputs[2], inputs[4]])
    squin.broadcast.sqrt_y_adj([inputs[0], inputs[3]])
    squin.broadcast.cz([inputs[0], inputs[2]], [inputs[1], inputs[3]])
    squin.broadcast.sqrt_x_adj([inputs[0], inputs[1], inputs[4]])


@kernel
def transversal_logical_msd(q0, q1, q2, q3, q4):
    squin.broadcast.sqrt_x(q0 + q1 + q4)
    squin.broadcast.cz(q0 + q2, q1 + q3)
    squin.broadcast.sqrt_y(q0 + q3)
    squin.broadcast.cz(q0 + q3, q2 + q4)
    squin.broadcast.sqrt_x_adj(q0)
    squin.broadcast.cz(q0 + q1, q4 + q3)
    squin.broadcast.sqrt_x_adj(q0 + q1 + q2 + q3 + q4)


@kernel
def special_x_phys():
    q = squin.qalloc(35)
    squin.broadcast.reset(q)
    q0 = q[0:7]
    q1 = q[7:14]
    q2 = q[14:21]
    q3 = q[21:28]
    q4 = q[28:35]
    inputs = [q0[6], q1[6], q2[6], q3[6], q4[6]]

    squin.h(q0[6])
    physical_msd_inverse(inputs)
    steane7_encode_existing(q0)
    steane7_encode_existing(q1)
    steane7_encode_existing(q2)
    steane7_encode_existing(q3)
    steane7_encode_existing(q4)
    transversal_logical_msd(q0, q1, q2, q3, q4)
    squin.broadcast.h(q0)
    squin.broadcast.measure(q)


@kernel
def special_y_phys():
    q = squin.qalloc(35)
    squin.broadcast.reset(q)
    q0 = q[0:7]
    q1 = q[7:14]
    q2 = q[14:21]
    q3 = q[21:28]
    q4 = q[28:35]
    inputs = [q0[6], q1[6], q2[6], q3[6], q4[6]]

    squin.h(q0[6])
    squin.sqrt_z(q0[6])
    physical_msd_inverse(inputs)
    steane7_encode_existing(q0)
    steane7_encode_existing(q1)
    steane7_encode_existing(q2)
    steane7_encode_existing(q3)
    steane7_encode_existing(q4)
    transversal_logical_msd(q0, q1, q2, q3, q4)
    squin.broadcast.sqrt_z_adj(q0)
    squin.broadcast.h(q0)
    squin.broadcast.measure(q)


@kernel
def special_z_phys():
    q = squin.qalloc(35)
    squin.broadcast.reset(q)
    q0 = q[0:7]
    q1 = q[7:14]
    q2 = q[14:21]
    q3 = q[21:28]
    q4 = q[28:35]
    inputs = [q0[6], q1[6], q2[6], q3[6], q4[6]]

    physical_msd_inverse(inputs)
    steane7_encode_existing(q0)
    steane7_encode_existing(q1)
    steane7_encode_existing(q2)
    steane7_encode_existing(q3)
    steane7_encode_existing(q4)
    transversal_logical_msd(q0, q1, q2, q3, q4)
    squin.broadcast.measure(q)


physical_special_kernels = {
    'X': special_x_phys,
    'Y': special_y_phys,
    'Z': special_z_phys,
}

In [ ]:
def summarize_hybrid_special_state(basis: str, shots: int = 256) -> dict[str, object]:
    raw_shots = Circuit(physical_special_kernels[basis]).compile_sampler(seed=1).sample(shots=shots)
    post = logical_tasks[basis]._post_processing

    detectors = np.asarray(list(post.emit_detectors(raw_shots.tolist())), dtype=np.uint8)
    observables = np.asarray(list(post.emit_observables(raw_shots.tolist())), dtype=np.uint8)

    return {
        'raw_measurement_shape': tuple(raw_shots.shape),
        'detector_shape': tuple(detectors.shape),
        'observable_shape': tuple(observables.shape),
        'all_zero_detectors': bool(np.all(detectors == 0)),
        'all_zero_observables': bool(np.all(observables == 0)),
        'unique_detector_rows': np.unique(detectors, axis=0).tolist(),
        'unique_observable_rows': np.unique(observables, axis=0).tolist(),
    }


HYBRID_RESULTS = {
    basis: summarize_hybrid_special_state(basis)
    for basis in ('X', 'Y', 'Z')
}

HYBRID_RESULTS

## Interpretation

What this prototype currently tells us:

- The hybrid construction is **detector-clean** in noiseless simulation under the logical suffix post-processing.
- The logical observables are **deterministic**, but under the current Bloqade logical post-processing they are **not yet `00000`**.
- So this notebook is a good staging point for the next step, but it is not yet the final paper-faithful special-state path.

The remaining mismatch is most likely one of:

1. qubit-order / output-logical convention,
2. the exact Clifford fake-input sequence versus the naive `msd_inverse` gate list,
3. Bloqade's observable convention versus the one used by `distillation_sim`.

That is exactly why this notebook is useful before integrating anything into the decoder notebook.